# Object Storage: Buckets and Objects

**Before running:** start `moto_server -p 5000` in a separate terminal and leave it open.


## Setup: connect to the local server

In [1]:
import os, hashlib
from datetime import datetime, timedelta, timezone
import boto3
from botocore.exceptions import ClientError

s3 = boto3.client(
    "s3",
    endpoint_url="http://127.0.0.1:5000",
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1",
)

## Question 1
Create bucket `hospital-scans`, list all buckets, print names.

In [2]:
s3.create_bucket(Bucket="hospital-scans")
for b in s3.list_buckets()["Buckets"]:
    print(b["Name"])

hospital-scans


## Question 2
Upload `xray-chest.png` to `patients/P-40231/2026-09-01/xray-chest.png`; print each key + size.

In [3]:
B = "hospital-scans"
K = "patients/P-40231/2026-09-01/xray-chest.png"
s3.upload_file("sample_files/xray-chest.png", B, K)
for o in s3.list_objects_v2(Bucket=B)["Contents"]:
    print(o["Key"], o["Size"])

patients/P-40231/2026-09-01/xray-chest.png 20000


## Question 3
Upload the other 3 scans with ContentType + metadata (`patient`, `scan-type`); print object count.

In [4]:
uploads = [
    ("lab-report.pdf", "patients/P-40231/2026-09-01/lab-report.pdf", "application/pdf",   "P-40231", "lab"),
    ("mri-brain.dcm",  "patients/P-40231/2026-09-15/mri-brain.dcm",  "application/dicom", "P-40231", "mri"),
    ("xray-wrist.png", "patients/P-40232/2026-09-03/xray-wrist.png", "image/png",         "P-40232", "xray"),
]
for f, key, ctype, patient, scan in uploads:
    s3.upload_file(f"sample_files/{f}", B, key,
                   ExtraArgs={"ContentType": ctype,
                              "Metadata": {"patient": patient, "scan-type": scan}})
print(s3.list_objects_v2(Bucket=B)["KeyCount"])

4


## Question 4
List all objects: print count, then each key.

In [5]:
r = s3.list_objects_v2(Bucket=B)
print(r["KeyCount"])
for o in r["Contents"]:
    print(o["Key"])

4
patients/P-40231/2026-09-01/lab-report.pdf
patients/P-40231/2026-09-01/xray-chest.png
patients/P-40231/2026-09-15/mri-brain.dcm
patients/P-40232/2026-09-03/xray-wrist.png


## Question 5
Prefix `patients/P-40231/`: print count, then key + size.

In [6]:
r = s3.list_objects_v2(Bucket=B, Prefix="patients/P-40231/")
print(r["KeyCount"])
for o in r["Contents"]:
    print(o["Key"], o["Size"])

3
patients/P-40231/2026-09-01/lab-report.pdf 6000
patients/P-40231/2026-09-01/xray-chest.png 20000
patients/P-40231/2026-09-15/mri-brain.dcm 50000


## Question 6
Delimiter `/` under `patients/`: print each CommonPrefix.

In [7]:
r = s3.list_objects_v2(Bucket=B, Prefix="patients/", Delimiter="/")
for c in r["CommonPrefixes"]:
    print(c["Prefix"])

patients/P-40231/
patients/P-40232/


## Question 7
head_object on the MRI: print size, content type, metadata.

In [8]:
h = s3.head_object(Bucket=B, Key="patients/P-40231/2026-09-15/mri-brain.dcm")
print(h["ContentLength"], h["ContentType"], h["Metadata"])

50000 application/dicom {'patient': 'P-40231', 'scan-type': 'mri'}


## Question 8
Download xray-chest to `downloads/`, compare MD5 with original.

In [9]:
os.makedirs("downloads", exist_ok=True)
s3.download_file(B, K, "downloads/xray-chest.png")

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

print(md5("downloads/xray-chest.png") == md5("sample_files/xray-chest.png"))

True


## Question 9
Overwrite with `xray-chest-v2.png`; print size before -> after and whether ETag changed.

In [10]:
before = s3.head_object(Bucket=B, Key=K)
s3.upload_file("sample_files/xray-chest-v2.png", B, K)
after = s3.head_object(Bucket=B, Key=K)
print(before["ContentLength"], "->", after["ContentLength"])
print("ETag changed:", before["ETag"] != after["ETag"])

20000 -> 22000
ETag changed: True


## Question 10
Move lab-report to `archive/P-40231/lab-report.pdf` (copy + delete); print all keys.

In [11]:
old = "patients/P-40231/2026-09-01/lab-report.pdf"
new = "archive/P-40231/lab-report.pdf"
s3.copy_object(Bucket=B, Key=new, CopySource={"Bucket": B, "Key": old})
s3.delete_object(Bucket=B, Key=old)
for o in s3.list_objects_v2(Bucket=B)["Contents"]:
    print(o["Key"])

archive/P-40231/lab-report.pdf
patients/P-40231/2026-09-01/xray-chest.png
patients/P-40231/2026-09-15/mri-brain.dcm
patients/P-40232/2026-09-03/xray-wrist.png


## Question 11
Enable versioning, re-upload original xray-chest; print version id, IsLatest, size.

In [12]:
s3.put_bucket_versioning(Bucket=B, VersioningConfiguration={"Status": "Enabled"})
s3.upload_file("sample_files/xray-chest.png", B, K)
for v in s3.list_object_versions(Bucket=B, Prefix=K)["Versions"]:
    print(v["VersionId"], v["IsLatest"], v["Size"])

24e95206-2404-4f12-a58c-9398c78797ca True 20000
null False 22000


## Question 12
Delete xray-wrist (a) delete marker? (b) visible count (c) versions/markers (d) restore + count.

In [13]:
W = "patients/P-40232/2026-09-03/xray-wrist.png"

# (a)
r = s3.delete_object(Bucket=B, Key=W)
print("(a)", r.get("DeleteMarker", False))

# (b)
print("(b)", s3.list_objects_v2(Bucket=B)["KeyCount"])

# (c)
r = s3.list_object_versions(Bucket=B, Prefix=W)
versions, markers = r.get("Versions", []), r.get("DeleteMarkers", [])
print("(c)", len(versions), "version and", len(markers), "delete marker")

# (d) deleting the delete marker brings the object back
s3.delete_object(Bucket=B, Key=W, VersionId=markers[0]["VersionId"])
print("(d)", s3.list_objects_v2(Bucket=B)["KeyCount"])

(a) True
(b) 3
(c) 1 version and 1 delete marker
(d) 4


## Question 13
Bucket `bank-statements` with object lock; COMPLIANCE 30 days; (b) mode/date (c) delete error (d) count.

In [14]:
BB = "bank-statements"
SK = "statements/acct-771/2026-03.pdf"
s3.create_bucket(Bucket=BB, ObjectLockEnabledForBucket=True)

# (a)
until = datetime.now(timezone.utc) + timedelta(days=30)
with open("sample_files/statement-2026-03.pdf", "rb") as f:
    r = s3.put_object(Bucket=BB, Key=SK, Body=f,
                      ObjectLockMode="COMPLIANCE",
                      ObjectLockRetainUntilDate=until)
vid = r["VersionId"]

# (b)
h = s3.head_object(Bucket=BB, Key=SK)
print("(b)", h["ObjectLockMode"], h["ObjectLockRetainUntilDate"])

# (c)
try:
    s3.delete_object(Bucket=BB, Key=SK, VersionId=vid)
    print("(c) deleted (lock did not work)")
except ClientError as e:
    print("(c)", e.response["Error"]["Code"])

# (d)
print("(d)", s3.list_objects_v2(Bucket=BB)["KeyCount"])

(b) COMPLIANCE 2026-11-18 11:22:53+00:00
(c) AccessDenied
(d) 1


## Question 14
Bucket `photo-app`; (a) list sizes (b) `get_photo()` (c) 50-photo scroll bytes + ratio.

In [15]:
PB = "photo-app"
s3.create_bucket(Bucket=PB)
for f, name in [("photo-original.jpg", "original"),
                ("photo-feed.jpg", "feed"),
                ("photo-thumb.jpg", "thumb")]:
    s3.upload_file(f"sample_files/{f}", PB, f"photos/8831/{name}.jpg",
                   ExtraArgs={"ContentType": "image/jpeg"})

# (a)
print("(a)")
for o in s3.list_objects_v2(Bucket=PB, Prefix="photos/8831/")["Contents"]:
    print(o["Key"], o["Size"])

# (b)
def get_photo(photo_id, screen):
    size = {"list": "thumb", "feed": "feed", "fullscreen": "original"}[screen]
    return s3.get_object(Bucket=PB, Key=f"photos/{photo_id}/{size}.jpg")["Body"].read()

print("(b)")
for screen in ["list", "feed", "fullscreen"]:
    print(screen, len(get_photo(8831, screen)))

# (c) sizes from head_object, nothing downloaded
feed_total = s3.head_object(Bucket=PB, Key="photos/8831/feed.jpg")["ContentLength"] * 50
orig_total = s3.head_object(Bucket=PB, Key="photos/8831/original.jpg")["ContentLength"] * 50
print("(c)")
print(f"feed copies: {feed_total} bytes")
print(f"originals: {orig_total} bytes")
print("ratio:", orig_total / feed_total)

(a)
photos/8831/feed.jpg 2000
photos/8831/original.jpg 80000
photos/8831/thumb.jpg 500
(b)
list 500
feed 2000
fullscreen 80000
(c)
feed copies: 100000 bytes
originals: 4000000 bytes
ratio: 40.0


## Question 15
Chunk `textbook.bin` at 4096 bytes; total, unique, saving %. (b) written answer.

In [16]:
hashes = []
with open("sample_files/textbook.bin", "rb") as f:
    while True:
        chunk = f.read(4096)
        if not chunk:
            break
        hashes.append(hashlib.md5(chunk).hexdigest())

total, unique = len(hashes), len(set(hashes))
print("Total chunks:", total)
print("Unique chunks:", unique)
print(f"Saving: {(total - unique) / total * 100:.1f}%")

Total chunks: 12
Unique chunks: 8
Saving: 33.3%


### Question 15(b): written answer

_Write your explanation here._